# Anomaly Detection (Isolation Forest)-Unsupervised

In [3]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [6]:
#load engineered features
video_features = pd.read_csv("data/processed/video_features_engineered.csv")

In [7]:
video_features.shape

(9999, 14)

In [19]:
# Select features for the model
feature_cols = [
    "like_view_ratio",
    "max_like_spike",
    "max_view_spike", 
    "like_spike_ratio",
    "view_spike_ratio",
    "peak_hour",
    "hours_tracked"
]

X = video_features[feature_cols].fillna(0)

# Scale features - important for Isolation Forest
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
# contamination = expected % of anomalies in data
# We estimate ~5% of videos have fake engagement
iso_forest = IsolationForest(contamination=0.08, random_state=42)
video_features["anomaly_score"] = iso_forest.fit_predict(X_scaled)
video_features["is_suspicious"] = video_features["anomaly_score"] == -1

print(f"Total videos analysed  : {len(video_features):,}")
print(f"Flagged as suspicious  : {video_features['is_suspicious'].sum():,}")
print(f"Suspicion rate         : {video_features['is_suspicious'].mean():.1%}")

Total videos analysed  : 9,999
Flagged as suspicious  : 800
Suspicion rate         : 8.0%


## Sustainability Feature

In [ ]:
### Measures whether engagement sustains AFTER the peak spike. Suspicious videos often have a big spike 
# followed by a sharp drop (dump & leave), while organic content has sustained engagement over time.
### Low sustainability = bot signature (dump & leave)
### High sustainability = organic viral content

In [11]:
yt_sample = pd.read_csv("data/processed/yt_sample.csv")

In [21]:
def add_sustainability(df):
    results = []
    
    for video_id, group in df.groupby("video_id"):
        group = group.sort_values("created_at")
        
        if len(group) < 3:
            continue
        
        like_growth = group["likes"].diff().fillna(0)
        
        # Find the hour of maximum spike
        spike_idx = like_growth.idxmax()
        spike_pos = group.index.get_loc(spike_idx)
        
        # Look at what happens AFTER the spike
        # If bot: likes drop back to near zero immediately
        # If viral: likes stay elevated for several hours
        if spike_pos + 3 < len(group):
            after_spike = like_growth.iloc[spike_pos+1 : spike_pos+4].mean()
            spike_value = like_growth.iloc[spike_pos]
            
            # Ratio of post-spike activity to spike itself
            # Low ratio = dropped immediately = bot signature
            sustainability = after_spike / (spike_value + 1)
        else:
            sustainability = 0
            
        results.append({
            "video_id"       : video_id,
            "sustainability" : sustainability
        })
    
    return pd.DataFrame(results)


sustainability_df = add_sustainability(yt_sample)
video_features = video_features.merge(sustainability_df, on="video_id", how="left")

print(f"Feature added ✓")
print(f"\nSustainability stats:")
print(video_features["sustainability"].describe())

Feature added ✓

Sustainability stats:
count    9999.000000
mean        0.134525
std         0.216955
min        -0.619048
25%         0.000000
50%         0.000000
75%         0.238492
max         0.952381
Name: sustainability, dtype: float64


In [13]:
# Merge back into feature dataframe
video_features = video_features.merge(sustainability_df, on="video_id", how="left")
print(f"Sustainability stats:")
print(video_features["sustainability"].describe())

Sustainability stats:
count    9999.000000
mean        0.134525
std         0.216955
min        -0.619048
25%         0.000000
50%         0.000000
75%         0.238492
max         0.952381
Name: sustainability, dtype: float64


In [28]:
# Compare suspicious vs normal
suspicious = video_features[video_features['is_suspicious']]
normal = video_features[~video_features['is_suspicious']]
print(f"Suspicious avg sustainability: {suspicious['sustainability'].mean():.3f}")
print(f"Normal avg sustainability : {normal['sustainability'].mean():.3f}")

Suspicious avg sustainability: 0.322
Normal avg sustainability : 0.118


In [29]:
video_features.head()

,video_id,total_views,total_likes,total_comments,total_dislikes,like_view_ratio,comment_view_ratio,dislike_view_ratio,max_like_spike,max_view_spike,...,view_spike_ratio,peak_hour,hours_tracked,anomaly_score,is_suspicious,sustainability_x,sustainability_y,sustainability,anomaly_score_v2,is_suspicious_v2
0,100012,3924,52.0,10.0,3.0,0.013248,0.002548,0.000764,13.0,414.0,...,18.223718,18,170,1,False,0.142857,0.142857,0.142857,1,False
1,100087,22384,147.0,257.0,98.0,0.006567,0.011481,0.004378,13.0,2110.0,...,15.993401,0,170,1,False,0.666667,0.666667,0.666667,1,False
2,100088,5198,89.0,46.0,10.0,0.017119,0.008848,0.001923,13.0,988.0,...,32.772683,12,170,1,False,0.428571,0.428571,0.428571,1,False
3,100106,6094,93.0,6.0,2.0,0.015258,0.000984,0.000328,18.0,970.0,...,27.419355,19,170,1,False,0.280702,0.280702,0.280702,1,False
4,100119,34569,304.0,176.0,69.0,0.008794,0.005091,0.001996,58.0,5882.0,...,29.160421,17,170,1,False,0.338983,0.338983,0.338983,1,False


### Rerun the model with additional sustainability features

In [26]:
feature_cols = [
    "like_view_ratio",
    "max_like_spike",
    "max_view_spike",
    "like_spike_ratio",
    "view_spike_ratio",
    "peak_hour",
    "hours_tracked",
    "sustainability"
]

X = video_features[feature_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso_forest2 = IsolationForest(contamination=0.05, random_state=42)
video_features["anomaly_score_v2"] = iso_forest2.fit_predict(X_scaled)
video_features["is_suspicious_v2"] = video_features["anomaly_score_v2"] == -1

# Compare v1 vs v2 flagging
both_flagged    = ((video_features["is_suspicious"]) & (video_features["is_suspicious_v2"])).sum()
only_v1_flagged = ((video_features["is_suspicious"]) & (~video_features["is_suspicious_v2"])).sum()
only_v2_flagged = ((~video_features["is_suspicious"]) & (video_features["is_suspicious_v2"])).sum()

print(f"Flagged by both models    : {both_flagged}")
print(f"Dropped by v2 (v1 only)  : {only_v1_flagged}")
print(f"New flags in v2           : {only_v2_flagged}")

Flagged by both models    : 490
Dropped by v2 (v1 only)  : 310
New flags in v2           : 10


In [30]:
video_features.to_csv("data/processed/engineered_video_features2.csv", index=False)

In [31]:
video_features.head()

,video_id,total_views,total_likes,total_comments,total_dislikes,like_view_ratio,comment_view_ratio,dislike_view_ratio,max_like_spike,max_view_spike,...,view_spike_ratio,peak_hour,hours_tracked,anomaly_score,is_suspicious,sustainability_x,sustainability_y,sustainability,anomaly_score_v2,is_suspicious_v2
0,100012,3924,52.0,10.0,3.0,0.013248,0.002548,0.000764,13.0,414.0,...,18.223718,18,170,1,False,0.142857,0.142857,0.142857,1,False
1,100087,22384,147.0,257.0,98.0,0.006567,0.011481,0.004378,13.0,2110.0,...,15.993401,0,170,1,False,0.666667,0.666667,0.666667,1,False
2,100088,5198,89.0,46.0,10.0,0.017119,0.008848,0.001923,13.0,988.0,...,32.772683,12,170,1,False,0.428571,0.428571,0.428571,1,False
3,100106,6094,93.0,6.0,2.0,0.015258,0.000984,0.000328,18.0,970.0,...,27.419355,19,170,1,False,0.280702,0.280702,0.280702,1,False
4,100119,34569,304.0,176.0,69.0,0.008794,0.005091,0.001996,58.0,5882.0,...,29.160421,17,170,1,False,0.338983,0.338983,0.338983,1,False
